# 02_sae_training

Top-k Sparse Autoencoder trained on distilgpt2 layer-3 activations.

Converted from the provided Python script into notebook format.

## Header / Overview

In [ ]:
"""
02_sae_training.py
──────────────────
Top-k Sparse Autoencoder trained on distilgpt2 layer-3 activations.

Assumes Notebook 01 output is attached as a dataset at SHARD_DIR.
Copy the ShardedActivationBuffer and ActivationNormalizer classes from
01_pipeline_setup.py into this notebook, or paste them above this file.

Section map
  1. Config
  2. SAE model
  3. Training loop
  4. Orchestration
"""

## 1. Config

In [ ]:
# ── 1. Config ─────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from typing import Optional, Iterator

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
D_MODEL   = 768      # distilgpt2 hidden dim
M         = 1024      # SAE bottleneck width
K         = M // 10  # top-k sparsity: 10% of M active features per token

LR             = 1e-4
BATCH          = 4_096
N_STEPS        = 100_000
CKPT_EVERY     = 10_000
LOG_EVERY      = 100
DEAD_WINDOW    = 500   # steps without activation before a feature is "dead"

# Update this path after attaching Notebook 01's output dataset
SHARD_DIR = Path("/kaggle/input/notebooks/codemtc/pipeline-setup/activations")
OUT_DIR   = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Classes from 1st Notebook

In [ ]:
class ActivationNormalizer:
    """
    Per-dimension (channel-wise) z-score normalizer.

    Fit on the debug pass (~200k tokens); save and reuse for the full extraction
    and at eval time. Consistent normalization is critical: the SAE learns a
    dictionary in the normalized space, so test-time patching must use the same
    mean/std.

    Why per-dimension?  distilgpt2 layer-3 output dimensions have very different
    scales — some are near-zero throughout the dataset, others span ±10+.
    Global (scalar) normalization leaves that structure intact, making the SAE
    learn an uneven dictionary. Per-dimension normalization is what Anthropic and
    Nanda's public SAE implementations both use.
    """

    def __init__(self):
        self.mean: Optional[torch.Tensor] = None
        self.std:  Optional[torch.Tensor] = None

    def fit(self, acts: torch.Tensor, eps: float = 1e-6):
        """acts: (N, D_MODEL), any dtype."""
        a = acts.float()
        self.mean = a.mean(dim=0)                    # (D_MODEL,)
        self.std  = a.std(dim=0).clamp(min=eps)      # (D_MODEL,)
        print(
            f"  normalizer: mean.norm={self.mean.norm():.3f}  "
            f"std.mean={self.std.mean():.4f}  "
            f"std.min={self.std.min():.6f}"
        )

    def __call__(self, acts: torch.Tensor) -> torch.Tensor:
        """Return normalized activations in the same dtype as input."""
        dtype = acts.dtype
        a = acts.float()
        out = (a - self.mean.to(a.device)) / self.std.to(a.device)
        return out.to(dtype)

    def inverse(self, acts: torch.Tensor) -> torch.Tensor:
        dtype = acts.dtype
        a = acts.float()
        out = a * self.std.to(a.device) + self.mean.to(a.device)
        return out.to(dtype)

    def save(self, path: Path):
        torch.save({"mean": self.mean, "std": self.std}, path)
        print(f"  normalizer saved → {path}")

    @classmethod
    def load(cls, path: Path) -> "ActivationNormalizer":
        n = cls()
        ckpt = torch.load(path, map_location="cpu")
        n.mean, n.std = ckpt["mean"], ckpt["std"]
        return n

In [ ]:
SAE_BATCH = 4_096
class ShardedActivationBuffer:
    """
    Drop-in replacement for ActivationBuffer when activations were saved to
    disk in a previous Kaggle session (Notebook 01 → Notebook 02).

    Loads one shard at a time, shuffles it, and yields SAE batches.  Cycles
    through all shards indefinitely (wraps around), which is fine because the
    SAE only needs ~100 k steps × 4096 batch ≈ 400 M activation rows total —
    less than the 80 M you extracted (each row seen ≈ 5 times on average).
    """

    def __init__(
        self,
        shard_dir: Path,
        sae_batch_size: int = SAE_BATCH,
        device: str = DEVICE,
    ):
        self._paths = sorted(shard_dir.glob("shard_*.pt"))
        assert self._paths, f"No shards found in {shard_dir}"
        self._bsz    = sae_batch_size
        self._device = device
        self._idx    = 0
        self._buf: Optional[torch.Tensor] = None
        self._ptr    = 0
        self._load_next()

    def _load_next(self):
        path = self._paths[self._idx % len(self._paths)]
        self._idx += 1
        buf = torch.load(path, map_location="cpu")   # fp16, (shard_size, D_MODEL)
        # shuffle
        buf = buf[torch.randperm(len(buf))]
        self._buf = buf
        self._ptr = 0
        print(f"  shard loaded: {path.name}  ({len(self._buf):,} rows)")

    def __iter__(self):
        return self

    def __next__(self) -> torch.Tensor:
        if self._ptr + self._bsz > len(self._buf):
            self._load_next()
        batch = self._buf[self._ptr : self._ptr + self._bsz]
        self._ptr += self._bsz
        return batch.to(self._device, dtype=torch.float32, non_blocking=True)

In [ ]:
def _prepend(tensor: torch.Tensor, gen: Iterator) -> Iterator:
    """Utility: yield a single tensor before resuming a generator."""
    yield tensor
    yield from gen

## 2. SAE Model

In [ ]:
# ── 2. SAE model ──────────────────────────────────────────────────────────────
class SparseAutoencoder(nn.Module):
    """
    Top-k Sparse Autoencoder.

    Architecture (tied-bias formulation, standard in mech interp):
        z    = topk( ReLU( W_enc @ (x - b_dec) + b_enc ) )
        x̂   = W_dec @ z + b_dec
        loss = MSE(x, x̂)

    The pre-encoder bias (b_dec) is subtracted before encoding and added
    back after decoding. This centers the input around the decoder's
    natural origin so the encoder learns directions, not offsets.

    Decoder columns (dictionary atoms) are kept at unit norm after every
    gradient step. Without this constraint, the trivially optimal solution
    is to make decoder atoms very large and encoder weights very small,
    which achieves low MSE without learning anything meaningful.

    Top-k vs L1
    ───────────
    The assignment specifies top-k sparsity rather than the L1 penalty used
    in Anthropic's original paper. Top-k has two advantages here: (1) it
    guarantees exactly K active features per token rather than varying
    sparsity, making L0 a constant diagnostic rather than a tunable one;
    (2) it removes the L1 coefficient as a hyperparameter to tune.
    """

    def __init__(self, d_in: int = D_MODEL, m: int = M, k: int = K):
        super().__init__()
        self.d_in = d_in
        self.m    = m
        self.k    = k

        # Encoder weights and bias
        self.W_enc = nn.Parameter(torch.empty(d_in, m))
        self.b_enc = nn.Parameter(torch.zeros(m))

        # Decoder weights (columns = dictionary atoms) and shared bias
        self.W_dec = nn.Parameter(torch.empty(d_in, m))
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        self._init_weights()

    def _init_weights(self):
        nn.init.kaiming_uniform_(self.W_enc, nonlinearity="relu")
        # Initialize decoder as encoder transpose then normalize.
        # This gives a reasonable starting point where encoder and decoder
        # are approximately inverses of each other.
        with torch.no_grad():
            self.W_dec.data = self.W_enc.data.T.clone()
            self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        """Normalize each column of W_dec to unit norm in-place."""
        # W_dec: (m, d_in) — normalize along d_in dimension (dim=1)
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data /= norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, d_in) → z: (batch, m)
        z has exactly K non-zero entries per row.
        """
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc  # (batch, m)
        acts = F.relu(pre)

        # Hard top-k: scatter the k largest values, zero the rest.
        # sorted=False is faster and order doesn't matter here.
        topk_vals, topk_idx = acts.topk(self.k, dim=-1, sorted=False)
        z = torch.zeros_like(acts)
        z.scatter_(-1, topk_idx, topk_vals)
        return z

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """z: (batch, m) → x̂: (batch, d_in)"""
        return z @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor):
        z    = self.encode(x)
        x_hat = self.decode(z)
        loss  = F.mse_loss(x_hat, x)
        return loss, z, x_hat

## 3. Training Loop

In [ ]:
# ── 3. Training loop ──────────────────────────────────────────────────────────
def train(
    sae: SparseAutoencoder,
    buffer,                           # ShardedActivationBuffer or ActivationBuffer
    n_steps: int       = N_STEPS,
    lr: float          = LR,
    ckpt_every: int    = CKPT_EVERY,
    log_every: int     = LOG_EVERY,
    dead_window: int   = DEAD_WINDOW,
    out_dir: Path      = OUT_DIR,
    resume_from: Optional[str] = None,
) -> list[float]:
    """
    Train the SAE and return a list of (step, loss) pairs for plotting.

    Dead feature tracking
    ─────────────────────
    `steps_since_active` counts how many consecutive steps each feature
    has gone without firing (having a non-zero output on any token in the
    batch). Features dead for more than `dead_window` steps are reported.
    A healthy SAE has <5% dead features; >20% suggests the learning rate
    is too high or the bottleneck is too wide for the data.

    Decoder normalization
    ─────────────────────
    `sae._normalize_decoder()` is called after every optimizer step. Adam
    accumulates second-moment estimates that effectively scale each parameter's
    update — without re-normalization, decoder columns drift away from unit
    norm over training and the dictionary becomes redundant (several atoms
    pointing in the same direction at different magnitudes).
    """
    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)

    start_step = 0
    if resume_from:
        ckpt = torch.load(resume_from, map_location=DEVICE)
        sae.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_step = ckpt["step"] + 1
        print(f"  resumed from checkpoint at step {start_step:,}")

    # (m,) integer tensor: how many steps since each feature last fired
    steps_since_active = torch.zeros(sae.m, dtype=torch.long)

    log: list[tuple[int, float]] = []

    for step in range(start_step, n_steps):
        x = next(buffer)   # (batch, d_in), fp32, on GPU

        optimizer.zero_grad(set_to_none=True)
        loss, z, _ = sae(x)
        loss.backward()

        # Gradient clipping: prevents the decoder from being pulled
        # strongly in one direction by a single outlier batch.
        nn.utils.clip_grad_norm_(sae.parameters(), max_norm=1.0)

        optimizer.step()
        sae._normalize_decoder()   # must happen after every optimizer step

        # ── Dead feature tracking (on CPU, no GPU sync needed) ──
        fired = (z.detach() > 0).any(dim=0).cpu()   # (m,) bool
        steps_since_active[fired]  = 0
        steps_since_active[~fired] += 1

        # ── Logging ─────────────────────────────────────────────
        if step % log_every == 0:
            loss_val = loss.item()
            # L0: average number of active features per token (should ≈ K)
            l0       = (z.detach() > 0).float().sum(dim=-1).mean().item()
            dead_pct = (steps_since_active > dead_window).float().mean().item() * 100

            log.append((step, loss_val))
            print(
                f"  step {step:6d} | loss {loss_val:.6f} | "
                f"L0 {l0:5.1f} / {sae.k} | dead {dead_pct:4.1f}%"
            )

        # ── Checkpointing ────────────────────────────────────────
        if step % ckpt_every == 0 and step > start_step:
            _save_checkpoint(sae, optimizer, step, out_dir)

    # Final save
    _save_checkpoint(sae, optimizer, n_steps - 1, out_dir, tag="final")
    return log


def _save_checkpoint(sae, optimizer, step, out_dir, tag=None):
    label = tag or f"{step:06d}"
    path  = out_dir / f"sae_m{sae.m}_ckpt_{label}.pt"
    torch.save(
        {
            "step":           step,
            "model_state":    sae.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config":         {"d_in": sae.d_in, "m": sae.m, "k": sae.k},
        },
        path,
    )
    print(f"  checkpoint → {path.name}")


def load_sae(path: str | Path, device: str = DEVICE) -> SparseAutoencoder:
    """Load a saved SAE checkpoint. Use this at the top of Notebooks 03 & 04."""
    ckpt = torch.load(path, map_location=device)
    cfg  = ckpt["config"]
    sae  = SparseAutoencoder(d_in=cfg["d_in"], m=cfg["m"], k=cfg["k"]).to(device)
    sae.load_state_dict(ckpt["model_state"])
    sae.eval()
    print(f"  loaded SAE m={cfg['m']} k={cfg['k']} from step {ckpt['step']:,}")
    return sae

## 4. Orchestration

In [ ]:
# ── 4. Orchestration ──────────────────────────────────────────────────────────
if __name__ == "__main__":
    # ── Load normalizer + buffer ──────────────────────────────────────────────
    # (paste ActivationNormalizer and ShardedActivationBuffer from Notebook 01
    #  above this block, or import them if running as a module)
    normalizer = ActivationNormalizer.load(SHARD_DIR / "normalizer.pt")
    buf        = ShardedActivationBuffer(SHARD_DIR, sae_batch_size=BATCH)

    # ── Smoke test: confirm buffer feeds correct shapes ───────────────────────
    sample = next(buf)
    assert sample.shape == (BATCH, D_MODEL), f"unexpected shape: {sample.shape}"
    assert sample.device.type == DEVICE
    print(f"  buffer OK — batch shape {sample.shape}  mean {sample.mean():.4f}")

    # ── Build SAE ─────────────────────────────────────────────────────────────
    sae = SparseAutoencoder(d_in=D_MODEL, m=M, k=K).to(DEVICE)
    print(f"  SAE: d_in={D_MODEL}  m={M}  k={K}  "
          f"params={sum(p.numel() for p in sae.parameters()):,}")

    # ── Sanity check: one forward pass before committing to a full run ────────
    with torch.no_grad():
        loss_0, z_0, _ = sae(sample)
    print(f"  initial loss {loss_0.item():.4f}  "
          f"L0 {(z_0 > 0).float().sum(-1).mean():.1f}")
    # L0 should be exactly K (=M//10) if top-k is working correctly.
    assert abs((z_0 > 0).float().sum(-1).mean().item() - K) < 1, \
        "L0 is not K — check top-k implementation"

    # ── Train ─────────────────────────────────────────────────────────────────
    # To resume from a checkpoint:
    #   resume_from = "/kaggle/working/sae_m512_ckpt_050000.pt"
    # Otherwise leave as None.
    history = train(
        sae,
        buf,
        n_steps    = N_STEPS,
        lr         = LR,
        ckpt_every = CKPT_EVERY,
        log_every  = LOG_EVERY,
        resume_from = None,
    )

    # ── Quick loss plot (viewable in interactive mode) ────────────────────────
    try:
        import matplotlib.pyplot as plt
        steps, losses = zip(*history)
        plt.figure(figsize=(10, 4))
        plt.plot(steps, losses)
        plt.xlabel("step")
        plt.ylabel("MSE loss")
        plt.title(f"SAE m={M} k={K} training loss")
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"sae_m{M}_loss.png", dpi=120)
        plt.show()
    except Exception:
        pass

# Smoke Test

In [ ]:
# from pathlib import Path
# import torch

# # ── Test 1: shards exist ──────────────────────────────────────
# shards = sorted(SHARD_DIR.glob("shard_*.pt"))
# print(f"shards found: {len(shards)}")
# assert len(shards) == 10, "wrong number of shards — check SHARD_DIR"

# # ── Test 2: normalizer loads and looks reasonable ─────────────
# normalizer = ActivationNormalizer.load(SHARD_DIR / "normalizer.pt")
# print(f"normalizer mean norm : {normalizer.mean.norm():.3f}  (expect ~65)")
# print(f"normalizer std  mean : {normalizer.std.mean():.4f}  (expect ~2.3)")
# assert normalizer.mean.shape == (D_MODEL,)
# assert normalizer.std.min() > 0

# # ── Test 3: buffer feeds correct batches ──────────────────────
# buf = ShardedActivationBuffer(SHARD_DIR, sae_batch_size=BATCH)
# batch = next(buf)
# print(f"\nbatch shape  : {batch.shape}   (expect ({BATCH}, {D_MODEL}))")
# print(f"batch device : {batch.device}  (expect cuda)")
# print(f"batch mean   : {batch.mean():.4f}  (expect ~0)")
# print(f"batch std    : {batch.std():.4f}   (expect ~1)")
# assert batch.shape == (BATCH, D_MODEL)
# assert batch.device.type == "cuda"
# assert abs(batch.mean().item()) < 0.1,  "mean too far from 0 — normalizer issue"
# assert abs(batch.std().item()  - 1) < 0.2, "std too far from 1 — normalizer issue"

# # ── Test 4: SAE forward pass ──────────────────────────────────
# sae = SparseAutoencoder(d_in=D_MODEL, m=M, k=K).to(DEVICE)
# with torch.no_grad():
#     loss_0, z_0, _ = sae(batch)

# l0 = (z_0 > 0).float().sum(dim=-1).mean().item()
# print(f"\ninitial loss : {loss_0.item():.4f}  (expect 1–15)")
# print(f"L0           : {l0:.1f}          (expect exactly {K})")
# assert abs(l0 - K) < 0.5, f"L0 is {l0}, expected {K} — top-k is broken"

# print("\n✓ all smoke tests passed")

In [ ]:
# sae_test = SparseAutoencoder(d_in=D_MODEL, m=M, k=K).to(DEVICE)
# buf_test  = ShardedActivationBuffer(SHARD_DIR, sae_batch_size=BATCH)

# log_test = train(
#     sae_test,
#     buf_test,
#     n_steps    = 500,
#     lr         = LR,
#     ckpt_every = 99999,   # no checkpoints in the trial run
#     log_every  = 50,
#     out_dir    = OUT_DIR,
# )